# Boltz-2 screen for human GHS-R\n\nThis notebook runs the end-to-end GHS-R ligand screening workflow.

In [27]:
!pip install -U boltz pandas matplotlib pyyaml py3Dmol

  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)


In [28]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Keep Boltz cache on the Colab VM disk, not Google Drive. Drive can raise
# OSError: [Errno 5] Input/output error on many small files (e.g. mols/*.pkl).
os.environ['BOLTZ_CACHE'] = '/content/boltz_cache'
os.makedirs(os.environ['BOLTZ_CACHE'], exist_ok=True)
print('BOLTZ_CACHE =', os.environ['BOLTZ_CACHE'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BOLTZ_CACHE = /content/boltz_cache


In [29]:
# Clone https://github.com/2000calories/ghsr_af3_molecule_screening (skip if already present).
# Alternative: upload your own zip and unzip into /content/ghsr_af3_molecule_screening.
import os
import subprocess

_repo_dir = '/content/ghsr_af3_molecule_screening'
_clone_url = 'https://github.com/2000calories/ghsr_af3_molecule_screening.git'
if os.path.isdir(os.path.join(_repo_dir, '.git')):
    print('Repo already present:', _repo_dir)
else:
    subprocess.run(['git', 'clone', _clone_url, _repo_dir], check=True)


Repo already present: /content/ghsr_af3_molecule_screening


In [30]:
import os; os.chdir('/content/ghsr_af3_molecule_screening')

In [31]:
!python scripts/build_inputs.py --library data/ligand_library.csv --target-fasta data/target_ghsr.fasta --out-dir inputs

Wrote 1 YAML files to inputs
Skipped 3 entries with missing/invalid fields


In [32]:
# run_screen defaults to --accelerator auto (GPU if CUDA, else CPU — avoids errors on CPU-only Colab).
!python scripts/run_screen.py --inputs-dir inputs --outputs-dir outputs --recycling-steps 3 --diffusion-samples 1 --msa-mode server

[RUN ] sulforaphane
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Checking input data.
All inputs are already processed.
Processing 0 inputs with 0 threads.
0it [00:00, ?it/s]
Traceback (most recent call last):
  File "/usr/local/bin/boltz", line 8, in <module>
    sys.exit(cli())
             ^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1157, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1078, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1688, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1434, in invoke
    return ctx.invoke(self.callback, **ctx.params)
          

In [33]:
!python scripts/parse_results.py --library data/ligand_library.csv --outputs-dir outputs --results-dir results
!python scripts/visualize.py --ranking-csv results/ranking.csv --outputs-dir outputs --plots-dir results/plots --top-poses-dir results/top_poses
!zip -r outputs.zip outputs

Wrote results/ranking.csv
Wrote plots to results/plots and top poses to results/top_poses
updating: outputs/ (stored 0%)
updating: outputs/sulforaphane/ (stored 0%)
updating: outputs/sulforaphane/boltz_results_sulforaphane/ (stored 0%)
updating: outputs/sulforaphane/boltz_results_sulforaphane/msa/ (stored 0%)
updating: outputs/sulforaphane/boltz_results_sulforaphane/predictions/ (stored 0%)
updating: outputs/sulforaphane/boltz_results_sulforaphane/processed/ (stored 0%)
updating: outputs/sulforaphane/boltz_results_sulforaphane/processed/records/ (stored 0%)
updating: outputs/sulforaphane/boltz_results_sulforaphane/processed/msa/ (stored 0%)
updating: outputs/sulforaphane/boltz_results_sulforaphane/processed/mols/ (stored 0%)
updating: outputs/sulforaphane/boltz_results_sulforaphane/processed/templates/ (stored 0%)
updating: outputs/sulforaphane/boltz_results_sulforaphane/processed/structures/ (stored 0%)
updating: outputs/sulforaphane/boltz_results_sulforaphane/processed/constraints/ (

In [34]:
import pandas as pd
df = pd.read_csv('results/ranking.csv')
df.head(20)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import py3Dmol
from pathlib import Path
top_pose_dir = Path('results/top_poses')
poses = sorted(top_pose_dir.glob('*.pdb'))
print('Found', len(poses), 'pose files')
if poses:
  model = poses[0].read_text()
  view = py3Dmol.view(width=900, height=600)
  view.addModel(model, 'pdb')
  view.setStyle({'cartoon': {'color': 'spectrum'}})
  view.setStyle({'resn': ['LIG']}, {'stick': {}})
  view.zoomTo()
  view.show()